In [6]:
import torch
import itertools
import os
import pyro
import pyro.distributions as dist
from pyro.infer import Trace_ELBO
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from einops import repeat

from pyro_cases.base_vae import BaseVAEwRegister
from pyro_cases.run import vae_dict

In [7]:
output_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_refer_test_sample_dict/")
output_dir.mkdir(exist_ok=True)

In [8]:
device = torch.device("cuda:0")

In [9]:
n_test_obs = 1000
test_seed = 7272

In [10]:
for k, vae in vae_dict.items():
    refer_vae = vae(hidden_dim=1, use_neural_network=False).to(device=device)
    if isinstance(refer_vae, BaseVAEwRegister):
        refer_vae.do_register(n_test_obs)
    pyro.set_rng_seed(test_seed)
    test_sample_dict = refer_vae.generate_sample_dict(batch_size=n_test_obs)
    torch.save({
        tk: tv.cpu() if isinstance(tv, torch.Tensor) else tv
        for tk, tv in test_sample_dict.items()
    }, output_dir / f"test_sample_dict_{k}.pt")